<a href="https://colab.research.google.com/github/rvigneshwaran20078/ML-project/blob/main/Ml_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1.Setup and Preprocessing**

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, silhouette_score, calinski_harabasz_score, davies_bouldin_score
)

from scipy.cluster.hierarchy import dendrogram, linkage



df = pd.read_csv("/content/retail_sales_with_outliers_missing.csv")

# Identify column types
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include="object").columns
date_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns

# Imputers
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

# Numeric imputation
for col in num_cols:
    df[col] = num_imputer.fit_transform(df[[col]])

# Categorical imputation
for col in cat_cols:
    df[col] = cat_imputer.fit_transform(df[[col]]).ravel()

# Convert any date-like column automatically
for col in df.columns:
    if "date" in col.lower():
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Outlier treatment
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# Recency feature (auto-detect date column)
date_col = [c for c in df.columns if "date" in c.lower()][0]
df["Recency"] = (pd.to_datetime("today") - df[date_col]).dt.days

# Frequency feature (auto-detect customer column)
cust_col = [c for c in df.columns if "customer" in c.lower()][0]

# Monetary feature (auto-detect amount/total/sales column)
monetary_col = [c for c in df.columns if "amount" in c.lower() or "sales" in c.lower()][0]

# Build RFM table
rfm = df.groupby(cust_col).agg({
    "Recency":"min",
    cust_col:"count",
    monetary_col:"sum"
}).rename(columns={cust_col:"Frequency", monetary_col:"Monetary"})

# Scale RFM features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

print("RFM table ready:")
print(rfm.head())




FileNotFoundError: [Errno 2] No such file or directory: '/content/retail_sales_with_outliers_missing.csv'

In [ ]:


# --- Assume you already built rfm and rfm_scaled from your preprocessing pipeline ---

results = {}

def evaluate_classification(true_labels, pred_labels):
    """Evaluate clustering predictions against true labels."""
    conf_mat = confusion_matrix(true_labels, pred_labels)
    mapping = {}
    for cluster_id in range(conf_mat.shape[1]):
        mapping[cluster_id] = np.argmax(conf_mat[:, cluster_id])
    mapped_preds = pd.Series(pred_labels).map(mapping)

    return {
        "Accuracy": accuracy_score(true_labels, mapped_preds),
        "Precision": precision_score(true_labels, mapped_preds, average='macro'),
        "Recall": recall_score(true_labels, mapped_preds, average='macro'),
        "F1": f1_score(true_labels, mapped_preds, average='macro')
    }

def evaluate_unsupervised(X, labels):
    """Evaluate clustering predictions without true labels."""
    if len(set(labels)) > 1 and len(set(labels) - {-1}) > 1:
        return {
            "Silhouette": silhouette_score(X, labels),
            "Calinski-Harabasz": calinski_harabasz_score(X, labels),
            "Davies-Bouldin": davies_bouldin_score(X, labels)
        }
    else:
        return {"Silhouette": None, "Calinski-Harabasz": None, "Davies-Bouldin": None}

# --- Run models ---
models = {
    "KMeans": KMeans(n_clusters=4, random_state=42),
    "Agglomerative": AgglomerativeClustering(n_clusters=4),
    "DBSCAN": DBSCAN(eps=2.0, min_samples=5),
    "GaussianMixture": GaussianMixture(n_components=4, random_state=42),
    "PCA+KMeans": KMeans(n_clusters=4, random_state=42)
}

# PCA for dimensionality reduction
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)

for name, model in models.items():
    if name == "PCA+KMeans":
        labels = model.fit_predict(rfm_pca)
        X = rfm_pca
    else:
        labels = model.fit_predict(rfm_scaled)
        X = rfm_scaled

    # Choose evaluation mode
    if "TrueSegment" in df.columns:  # supervised metrics
        results[name] = evaluate_classification(df['TrueSegment'], labels)
    else:  # unsupervised metrics
        results[name] = evaluate_unsupervised(X, labels)

# --- Show comparison ---
metrics_df = pd.DataFrame(results).T
print(metrics_df)

**2.K-Means Clustering**

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster_KMeans'] = kmeans.fit_predict(rfm_scaled)
results['KMeans'] = evaluate_unsupervised(rfm_scaled, rfm['Cluster_KMeans'])

print("KMeans Silhouette:", silhouette_score(rfm_scaled, rfm['Cluster_KMeans']))

**3.Hierarchical Clustering**

In [ ]:
agg = AgglomerativeClustering(n_clusters=4)
rfm['Cluster_Hierarchical'] = agg.fit_predict(rfm_scaled)
print("Hierarchical Silhouette:", silhouette_score(rfm_scaled, rfm['Cluster_Hierarchical']))

linked = linkage(rfm_scaled, 'ward')
plt.figure(figsize=(10, 6))
dendrogram(linked, truncate_mode='level', p=5)
plt.title("Hierarchical Clustering Dendrogram")
plt.show()

**4.DBSCAN**

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
rfm['Cluster_DBSCAN'] = dbscan.fit_predict(rfm_scaled)

labels = rfm['Cluster_DBSCAN']
unique_labels = np.unique(labels)
print("Unique labels:", unique_labels)

# Compute silhouette score only if valid
if len(set(labels)) > 1 and len(set(labels) - {-1}) > 1:
    results['DBSCAN'] = evaluate_unsupervised( rfm_scaled,labels_db)
    print("DBSCAN Silhouette:", silhouette_score(rfm_scaled, labels))
else:
    results['DBSCAN'] = {"Accuracy": None, "Precision": None, "Recall": None, "F1": None}
    print("DBSCAN did not form enough clusters for silhouette scoring.")

# Plot clusters
plt.scatter(rfm_scaled[:,0], rfm_scaled[:,1], c=labels, cmap="plasma")
plt.title("DBSCAN Clusters")
plt.show()



**5.Gaussian Mixture Model**

In [ ]:
gmm = GaussianMixture(n_components=4, random_state=42)
rfm['Cluster_GMM'] = gmm.fit_predict(rfm_scaled)
results['GaussianMixture'] = evaluate_unsupervised(rfm_scaled,rfm['Cluster_GMM'],)
print("GMM Silhouette:", silhouette_score(rfm_scaled, rfm['Cluster_GMM']))

plt.scatter(rfm_scaled[:,0], rfm_scaled[:,1], c=rfm['Cluster_GMM'], cmap="coolwarm")
plt.title("Gaussian Mixture Clusters")
plt.show()

**6.PCA + K-Means**

In [ ]:
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)
kmeans_pca = KMeans(n_clusters=4, random_state=42)
rfm['Cluster_PCA_KMeans'] = kmeans_pca.fit_predict(rfm_pca)
results['PCA+KMeans'] = evaluate_unsupervised(rfm_scaled, rfm['Cluster_PCA_KMeans'])
print("PCA+KMeans Silhouette:", silhouette_score(rfm_pca, rfm['Cluster_PCA_KMeans']))

plt.scatter(rfm_pca[:,0], rfm_pca[:,1], c=rfm['Cluster_PCA_KMeans'], cmap="viridis")
plt.title("PCA + KMeans Clusters")
plt.show()

**Show Comparison**

In [ ]:
metrics_df = pd.DataFrame(results).T
print(metrics_df)
